In [1]:
# Configuration and imports
import sys
from pathlib import Path
import yaml

ROOT = Path.cwd()
for _ in range(3):
    if (ROOT / 'src').is_dir():
        break
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SETTINGS = ROOT / 'settings'
REGISTRY = SETTINGS / 'data_registry_base.yaml'
FARM_PROFILES = SETTINGS / 'farm_profile_base.yaml'

# Sizing objective: minimize_cost | minimize_energy | minimize_draw
OBJECTIVE = 'minimize_cost'

In [2]:
# Compute irrigation demand from farm profiles
from src.irrigation_demand import compute_irrigation_demand

demand_df = compute_irrigation_demand(
    farm_profiles_path=FARM_PROFILES,
    registry_path=REGISTRY,
)

active = demand_df[demand_df['total_demand_m3'] > 0]
print(f'Demand profile: {len(demand_df)} days ({len(active)} active, {len(demand_df) - len(active)} fallow)')
print(f'Date range: {demand_df["day"].min().date()} to {demand_df["day"].max().date()}')
print()
print(f'Mean active demand:  {active["total_demand_m3"].mean():.0f} m3/day')
print(f'P95 active demand:   {active["total_demand_m3"].quantile(0.95):.0f} m3/day')
print(f'Max daily demand:    {demand_df["total_demand_m3"].max():.0f} m3/day')
print(f'Strictest crop TDS:  {demand_df["crop_tds_requirement_ppm"].dropna().min():.0f} ppm')

Demand profile: 5479 days (5195 active, 284 fallow)
Date range: 2010-01-01 to 2024-12-31

Mean active demand:  215 m3/day
P95 active demand:   483 m3/day
Max daily demand:    732 m3/day
Strictest crop TDS:  510 ppm


In [3]:
# Run from-scratch sizing — recommends all components
from src.water_sizing import size_water_system

result = size_water_system(
    demand_df,
    REGISTRY,
    objective=OBJECTIVE,
)

s = result['summary']
m = s.get('metrics', {})

print('=== Recommended System ===')
print(f'Wells:       {s["n_wells"]} at depths {s["well_depths_m"]}')
print(f'Treatment:   {s["treatment_throughput_m3_hr"]:.1f} m3/hr  ({s["treatment_throughput_m3_hr"] * 24:.0f} m3/day rated)')
print(f'Storage:     {s["storage_capacity_m3"]} m3  ({s["storage_type"]})')
print(f'Feed factor: {s["feed_factor"]:.3f}  (f_treat={s["f_treat"]:.3f})')
print(f'Blended TDS: {s["blended_well_tds_ppm"]:.0f} ppm')
print()
print('=== Validation Metrics ===')
print(f'Deficit:     {m["deficit_fraction"]:.4f}')
print(f'Cost/m3:     ${m["cost_per_m3_delivered"]:.2f}')
print(f'Energy/m3:   {m["energy_per_m3_delivered"]:.2f} kWh')
print(f'GW fraction: {m["gw_fraction"]:.1%}')
print(f'Muni fraction: {m["municipal_fraction"]:.1%}')
print(f'CAPEX:       ${m["total_capex"]:,.0f}')
print(f'Annual OPEX: ${m["annual_opex"]:,.0f}')

=== Recommended System ===
Wells:       3 at depths [10, 20, 30]
Treatment:   15.0 m3/hr  (360 m3/day rated)
Storage:     2000 m3  (reservoir)
Feed factor: 1.197  (f_treat=0.897)
Blended TDS: 1467 ppm

=== Validation Metrics ===
Deficit:     0.0000
Cost/m3:     $0.31
Energy/m3:   0.41 kWh
GW fraction: 100.5%
Muni fraction: 13.3%
CAPEX:       $275,500
Annual OPEX: $27,333


In [4]:
# YAML output — copy into settings/water_systems_base.yaml
config = result['config']
config['config_name'] = 'baseline_water_systems'
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

config_name: baseline_water_systems
systems:
- name: main_irrigation
  wells:
  - name: well_1
    depth_m: 10
    tds_ppm: 1000
    pump_type: small_submersible
  - name: well_2
    depth_m: 20
    tds_ppm: 1400
    pump_type: small_submersible
  - name: well_3
    depth_m: 30
    tds_ppm: 2000
    pump_type: small_submersible
  treatment:
    type: bwro
    throughput_m3_hr: 15.0
    goal_output_tds_ppm: 400
  municipal_source:
    tds_ppm: 200
    cost_per_m3: 0.5
    throughput_m3_hr: 200
  storage:
    type: reservoir
    capacity_m3: 2000
    initial_level_m3: 1000
    max_output_m3_hr: 50
    initial_tds_ppm: 400



In [5]:
# Treatment-anchored optimization — uses the from-scratch throughput as anchor
"""Sizes wells, storage, and municipal around the recommended treatment
plant to maximize utilization in the 70-85% sweet spot."""
from src.water_sizing import optimize_water_system

recommended_throughput = s['treatment_throughput_m3_hr']

opt = optimize_water_system(
    demand_df,
    REGISTRY,
    treatment_throughput_m3_hr=recommended_throughput,
    target_utilization=0.80,
    objective=OBJECTIVE,
)

os = opt['summary']
om = os.get('metrics', {})
u = opt['utilization_metrics']

print(f'=== Optimized System (anchored at {recommended_throughput:.1f} m3/hr) ===')
print(f'Wells:       {os["n_wells"]} at depths {os["well_depths_m"]}')
print(f'Storage:     {os["storage_capacity_m3"]} m3  ({os["storage_type"]})')
print(f'Feed factor: {os["feed_factor"]:.3f}')
print(f'Deficit:     {om["deficit_fraction"]:.4f}')
print(f'Cost/m3:     ${om["cost_per_m3_delivered"]:.2f}')
print(f'Energy/m3:   {om["energy_per_m3_delivered"]:.2f} kWh')
print()
print('=== Utilization ===')
print(f'Active treatment days: {u["active_treatment_days"]}')
print(f'Avg utilization:  {u["avg_utilization_pct"]:.1f}%')
print(f'P95 utilization:  {u["p95_utilization_pct"]:.1f}%')
print(f'Days in sweet spot (70-85%): {u["days_in_sweet_spot"]}  ({u["sweet_spot_fraction"]:.1%})')
print(f'Days below: {u["days_below_sweet_spot"]},  Days above: {u["days_above_sweet_spot"]}')
if 'avg_energy_multiplier' in u:
    print(f'Avg energy multiplier:    {u["avg_energy_multiplier"]:.3f}')
    print(f'Avg maintenance mult:     {u["avg_maintenance_multiplier"]:.3f}')
    print(f'Avg membrane life mult:   {u["avg_membrane_life_multiplier"]:.3f}')

=== Optimized System (anchored at 15.0 m3/hr) ===
Wells:       3 at depths [10, 20, 30]
Storage:     2000 m3  (reservoir)
Feed factor: 1.197
Deficit:     0.0000
Cost/m3:     $0.32
Energy/m3:   0.44 kWh

=== Utilization ===
Active treatment days: 5166
Avg utilization:  46.3%
P95 utilization:  91.4%
Days in sweet spot (70-85%): 88  (1.7%)
Days below: 3981,  Days above: 1097
Avg energy multiplier:    1.181
Avg maintenance mult:     1.046
Avg membrane life mult:   1.058


In [6]:
# Side-by-side comparison: current config vs recommended
import pandas as pd

current = yaml.safe_load(open(SETTINGS / 'water_systems_base.yaml'))
cur = current['systems'][0]
rec = config['systems'][0]

rows = [
    ('Wells', len(cur['wells']), len(rec['wells'])),
    ('Well depths (m)', [w['depth_m'] for w in cur['wells']], [w['depth_m'] for w in rec['wells']]),
    ('Well TDS (ppm)', [w['tds_ppm'] for w in cur['wells']], [w['tds_ppm'] for w in rec['wells']]),
    ('Treatment (m3/hr)', cur['treatment']['throughput_m3_hr'], rec['treatment']['throughput_m3_hr']),
    ('Treatment (m3/day)', cur['treatment']['throughput_m3_hr'] * 24, rec['treatment']['throughput_m3_hr'] * 24),
    ('Tank capacity (m3)', cur['storage']['capacity_m3'], rec['storage']['capacity_m3']),
    ('Tank type', cur['storage']['type'], rec['storage']['type']),
    ('Municipal (m3/hr)', cur['municipal_source']['throughput_m3_hr'], rec['municipal_source']['throughput_m3_hr']),
]

comp = pd.DataFrame(rows, columns=['Parameter', 'Current', 'Recommended'])
print(comp.to_string(index=False))

         Parameter            Current        Recommended
             Wells                  3                  3
   Well depths (m)      [20, 50, 100]       [10, 20, 30]
    Well TDS (ppm) [1400, 3500, 6500] [1000, 1400, 2000]
 Treatment (m3/hr)                 50               15.0
Treatment (m3/day)               1200              360.0
Tank capacity (m3)                500               2000
         Tank type   underground_tank          reservoir
 Municipal (m3/hr)                200                200
